# Zero-to-Hero: Production-Grade AI Task Planning & Productivity Agent

This notebook explains and demonstrates:
- Multi-agent LangGraph planning workflow
- Priority algorithms (Eisenhower, RICE, WSJF, etc.)
- Dependency-aware scheduling
- Persistent memory with SQLite + ChromaDB
- Reflection, recommendations, and analytics
- Structured output and tool-calling patterns

## 1) Architecture and Agent Graph

The agent is modeled as a deterministic LangGraph pipeline with typed state:
`Planner -> Scheduler -> Validator -> Reflection -> Memory -> Reporter`.

- **Planner**: parse messy input, extract tasks, assign priority scores
- **Scheduler**: assign start/end blocks with deadline-aware heuristics
- **Validator**: identify high-risk blocks and planning gaps
- **Reflection**: identify what worked/failed and recommend corrections
- **Memory**: persist operational and semantic traces
- **Reporter**: assemble final structured planning artifact

In [1]:
from task_planning_agent.config import load_config
from task_planning_agent.agent.service import PlanningService
from task_planning_agent.schemas import PriorityStrategy

## 2) Prompt/Input Normalization and Extraction

The extractor supports messy multi-line input with hints like deadlines, estimates, mentions, and project tags.
It converts unstructured text into typed `Task` objects with confidence and risk signals.

In [2]:
cfg = load_config()
service = PlanningService(cfg)
sample_input = '''
- Finalize roadmap deck by tomorrow 5pm #roadmap @sara 120min
- Sprint retrospective prep before friday 11am 60min
- Refactor API auth middleware next monday 180min
- Weekly team sync meeting friday 3pm 45min
'''

## 3) End-to-End Planning Run

Run full LangGraph pipeline and inspect structured schedule output fields:
task, priority, deadline, estimate, dependencies, start/end, confidence, reasoning, risk level.

In [3]:
report = service.plan(
    user_id='demo-user',
    raw_input=sample_input,
    strategy=PriorityStrategy.WSJF,
    timezone='Asia/Kolkata',
)
report

PlanReport(plan_id='d0f9e7ab-3393-4878-b272-3ced8428c58b', user_id='demo-user', generated_at=datetime.datetime(2026, 6, 30, 12, 55, 32, 468308), summary='Generated 4 blocks for 4 tasks', schedule=[ScheduleResponse(task='Weekly team sync meeting friday 3pm 45min', priority=66.67, deadline=None, estimated_duration=45, dependencies=[], suggested_start_time=datetime.datetime(2026, 6, 30, 9, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Kolkata')), suggested_end_time=datetime.datetime(2026, 6, 30, 9, 45, tzinfo=zoneinfo.ZoneInfo(key='Asia/Kolkata')), confidence=0.7000000000000001, reasoning='Extracted from messy input using parser heuristics. | Priority: WSJF weighted shortest job first | topo_rank=0', risk_level=<RiskLevel.LOW: 'low'>), ScheduleResponse(task='Sprint retrospective prep before friday 11am 60min', priority=50.0, deadline=None, estimated_duration=60, dependencies=[], suggested_start_time=datetime.datetime(2026, 6, 30, 10, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Kolkata')), suggested_end

## 4) Reflection and Recommendation Layer

Reflection captures misses/overruns/context switches, then recommendation agent suggests batching, automation, and deep-work protection.

In [4]:
report.reflections, [r.model_dump() for r in report.recommendations]

(['Protect first deep-work block from meetings.',
  'Cap daily context switches to <= 5.',
  'Re-estimate repeated overrun tasks by +20%.'],
 [{'category': 'batching',
   'suggestion': 'Batch admin tasks into one 45-minute block.',
   'impact': 'medium'},
  {'category': 'deep_work',
   'suggestion': 'Reserve 90-minute focus window in morning for high-priority items.',
   'impact': 'high'},
  {'category': 'automation',
   'suggestion': 'Automate recurring reminders for weekly report preparation.',
   'impact': 'medium'}])

## 5) Memory and Semantic Search

Each plan/task is persisted in SQLite and indexed in ChromaDB for semantic retrieval.

In [5]:
service.memory.semantic_search('deadline risk and sprint planning')[:3]

[{'document': '\n- Finalize roadmap deck by tomorrow 5pm #roadmap @sara 120min\n- Sprint retrospective prep before friday 11am 60min\n- Refactor API auth middleware next monday 180min\n- Weekly team sync meeting friday 3pm 45min\n',
  'metadata': {'user_id': 'demo-user', 'kind': 'plan'},
  'distance': 3.863729476928711},
 {'document': 'Weekly team sync meeting friday 3pm 45min\nWeekly team sync meeting friday 3pm 45min',
  'metadata': {'kind': 'task',
   'plan_id': 'd0f9e7ab-3393-4878-b272-3ced8428c58b',
   'user_id': 'demo-user'},
  'distance': 4.727659225463867},
 {'document': 'Sprint retrospective prep before friday 11am 60min\nSprint retrospective prep before friday 11am 60min',
  'metadata': {'kind': 'task',
   'plan_id': 'd0f9e7ab-3393-4878-b272-3ced8428c58b',
   'user_id': 'demo-user'},
  'distance': 4.784867286682129}]

## 6) Evaluation and Analytics

Analytics tracks completion, delay, planning accuracy, focus minutes, deep work, meetings, context switches, energy, and burnout.
These metrics are stored in DuckDB and logged to MLflow.

In [6]:
report.analytics

AnalyticsSnapshot(completed_tasks=0, completion_rate=0.0, average_delay_minutes=0.0, planning_accuracy=100.0, focus_time_minutes=0, deep_work_minutes=300, meetings_minutes=45, context_switches=3, energy_score=92.5, burnout_score=7.5, weekly_productivity_score=48.5)

## 7) FastAPI and Streamlit Integration

- FastAPI endpoints expose planning, search, history, calendar, preferences, and reports.
- Streamlit dashboard visualizes timeline, Gantt, kanban-like inbox, analytics, and memory search.

## 8) Prompt Engineering and Structured Output

The system enforces structured Pydantic schemas throughout pipeline boundaries to reduce hallucinations and improve deterministic behavior.
Model fallback chains are configurable per family (Llama3, Qwen3, Gemma3, Phi-4 Mini, Mistral, DeepSeek).